# Cómo construir sistemas inteligentes sobre LLMs: Una introducción a RAG y Agentes

Los Large Language Models (LLMs) han transformado la manera en que interactuamos con sistemas de inteligencia artificial. Hoy en día, modelos como Gemini, GPT o Claude son capaces de responder preguntas, resumir información, generar texto y mantener conversaciones relativamente complejas.

Sin embargo, los LLMs por sí solos presentan varias limitaciones importantes:

- no poseen acceso automático a documentos específicos o privados,
- pueden generar información incorrecta o alucinaciones,
- no mantienen una memoria persistente confiable,
- y poseen capacidades limitadas para planificar acciones o recopilar información de manera estratégica.

En aplicaciones reales, especialmente en dominios sensibles como salud, educación o apoyo a usuarios, normalmente no basta con utilizar un LLM “directamente”. En cambio, es necesario construir sistemas más complejos alrededor del modelo.

En este notebook exploraremos dos conceptos fundamentales en sistemas modernos basados en LLMs:

## Retrieval-Augmented Generation (RAG)

RAG permite conectar un LLM con fuentes externas de conocimiento, como:
- documentos PDF,
- bases de conocimiento,
- artículos,
- protocolos clínicos,
- o información privada de una organización.

La idea principal es recuperar información relevante antes de generar una respuesta, permitiendo que el sistema responda utilizando evidencia y contexto específicos.

## Agentes

Un agente basado en LLMs es un sistema capaz de:
- decidir qué hacer,
- utilizar herramientas,
- recopilar información faltante,
- y mantener cierto estado o memoria conversacional.

En lugar de simplemente responder preguntas, un agente puede interactuar de manera más activa y estratégica.

## Objetivo de la unidad

Construir sistemas conversacionales capaces de:

- acceder a conocimiento externo,
- responder preguntas usando documentos reales,
- recopilar información relevante,
- y tomar decisiones simples durante una interacción.

Como caso de estudio, utilizaremos ejemplos relacionados con apoyo conversacional a pacientes con enfermedades crónicas.

---

# Conexión a un LLM

In [ ]:
# Instalación de librerías necesarias
#
# openai:
#     Permite interactuar con modelos OpenAI.
#
# python-dotenv:
#     Permite cargar variables de entorno desde un archivo .env
#     (útil para no escribir claves API directamente en el código).

!pip install openai python-dotenv


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Configuración de la API

Para utilizar un LLM remoto como OpenAI, necesitamos autenticarnos utilizando una API Key.

Por razones de seguridad, no es recomendable escribir claves directamente en el notebook.  
En su lugar, utilizaremos un archivo `.env`.

In [ ]:
# Carga de variables de entorno

from dotenv import load_dotenv
import os

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

# Verificación básica
if api_key is None:
    print("API Key no encontrada.")
else:
    print("API Key cargada correctamente.")

API Key cargada correctamente.


In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

In [ ]:
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {
            "role": "user",
            "content": "¿Se puede usar la acupuntura para tratar la obesidad?"
        }
    ]
)

print(response.choices[0].message.content)

Sí, la acupuntura se ha utilizado como una terapia complementaria para el tratamiento de la obesidad. Algunos estudios sugieren que la acupuntura puede ayudar a reducir el apetito, mejorar el metabolismo y regular el equilibrio hormonal, lo que podría contribuir a la pérdida de peso. Sin embargo, la evidencia científica no es concluyente y la acupuntura no debe considerarse un tratamiento único o principal para la obesidad.

Es importante entender que la obesidad es una condición compleja que generalmente requiere un enfoque multifacético, incluyendo cambios en la dieta, ejercicio físico, modificación de hábitos y, en algunos casos, tratamiento médico o quirúrgico. La acupuntura puede ser un complemento útil dentro de un plan integral supervisado por profesionales de la salud.

Si estás interesado en la acupuntura para la obesidad, te recomiendo consultar con un médico o un profesional certificado en acupuntura, y asegurarte de que se utilice como parte de un programa integral de trata

## Problema

Los LLMs poseen conocimiento general aprendido durante entrenamiento.

Sin embargo, en aplicaciones reales frecuentemente necesitamos trabajar con:

- documentos privados,
- artículos recientes,
- protocolos específicos,
- bases de conocimiento organizacionales,
- o información clínica especializada.

Los modelos no poseen acceso automático a estos documentos.

Entonces surge una pregunta:

> ¿Cómo podemos conectar un LLM con conocimiento externo?

La respuesta es Retrieval-Augmented Generation (RAG).

---

# Arquitectura general de un sistema RAG

RAG significa **Retrieval-Augmented Generation**.

La idea principal consiste en combinar:

- un sistema de búsqueda de información (*retrieval*),
- con un Large Language Model (*generation*).

En lugar de pedirle al LLM que responda únicamente utilizando el conocimiento aprendido durante entrenamiento, primero recuperamos información relevante desde documentos externos y luego la incorporamos al prompt.

De esta manera, el modelo puede generar respuestas:
- más precisas,
- más específicas,
- más actualizadas,
- y basadas en evidencia documental.

## Componentes principales de un sistema RAG

### 1. Documentos

El sistema parte a partir de una fuente de conocimiento externa:

- PDFs,
- artículos,
- manuales,
- protocolos clínicos,
- FAQs,
- bases de conocimiento,
- páginas web,
- etc.

Estos documentos contienen la información que queremos que el sistema utilice al responder.

### 2. Chunking

Los documentos completos suelen ser demasiado largos para procesarlos directamente.

Por esta razón, normalmente se dividen en fragmentos más pequeños llamados *chunks*.

Por ejemplo:

```text
Documento completo
↓
Chunk 1
Chunk 2
Chunk 3
...
```

El tamaño de los chunks influye fuertemente en:
- la calidad de búsqueda,
- la precisión,
- y el contexto entregado al modelo.

### 3. Embeddings

Cada chunk es transformado en un vector numérico llamado embedding.

Un embedding intenta representar el significado semántico del texto.

La idea principal es:

> textos con significado similar tendrán embeddings cercanos en el espacio vectorial.

Esto permite realizar búsqueda semántica.

Por ejemplo:

- "abandono del tratamiento"
- "pérdida de adherencia"

podrían terminar relativamente cerca en el espacio vectorial aunque no compartan exactamente las mismas palabras.


### 4. Base vectorial

Los embeddings son almacenados en una base vectorial.

Esta base permite:
- almacenar vectores,
- indexarlos,
- y recuperar rápidamente los fragmentos más similares a una consulta.

Ejemplos populares:
- Chroma,
- FAISS,
- Pinecone,
- Weaviate.


### 5. Retrieval (búsqueda)

Cuando el usuario hace una pregunta:

```text
"¿Qué factores afectan la adherencia?"
```

la pregunta también se transforma en embedding.

Luego el sistema busca:
- cuáles chunks poseen embeddings más cercanos,
- es decir, cuáles fragmentos son semánticamente más relevantes.


### 6. Construcción del contexto

Los fragmentos recuperados se agregan al prompt enviado al LLM.

Por ejemplo:

```text
Contexto:
[chunk 1]
[chunk 2]
[chunk 3]

Pregunta:
¿Qué factores afectan la adherencia?
```

El modelo ahora responde utilizando información concreta proveniente de documentos reales.


### 7. Generación final

Finalmente, el LLM genera la respuesta utilizando:
- la pregunta del usuario,
- y el contexto recuperado.

Esto reduce:
- alucinaciones,
- respuestas genéricas,
- y errores factuales.


## Idea central de RAG

En un sistema RAG:

- el conocimiento especializado vive en los documentos,
- mientras que el LLM actúa principalmente como:
  - sintetizador,
  - generador de lenguaje,
  - e interfaz conversacional.

Esta separación es extremadamente importante en aplicaciones reales.

---

# Primera implementación de un sistema RAG con Langchain

LangChain es un framework de desarrollo en Python diseñado para facilitar la creación de aplicaciones basadas en modelos de lenguaje (LLMs). Su principal aporte es estructurar de forma modular componentes clave como prompts, cadenas de procesamiento (chains), agentes, memoria y conectores a fuentes de datos externas. Esto permite construir sistemas complejos —como chatbots, asistentes o pipelines de RAG— de manera más organizada y reproducible, integrando fácilmente modelos de lenguaje con bases de datos, APIs y motores de búsqueda.

- Langchain: https://www.langchain.com/

In [ ]:
!pip install -q \
langchain \
langchain-community \
langchain-openai \
langchain-chroma \
langchain-huggingface \
pypdf \
sentence-transformers \
chromadb


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Fuente de conocimiento utilizada

Para este ejemplo utilizaremos documentos provenientes de la Guía Clínica de Obesidad en Chile:

[Guía de Obesidad Chile](https://guiasobesidadchile.com/?utm_source=chatgpt.com)

Estas guías contienen recomendaciones clínicas oficiales relacionadas con:
- obesidad,
- tratamiento,
- adherencia,
- actividad física,
- alimentación,
- seguimiento de pacientes,
- y manejo multidisciplinario.

En aplicaciones reales, los sistemas RAG frecuentemente se construyen sobre:
- protocolos clínicos,
- documentación organizacional,
- bases de conocimiento privadas,
- o literatura científica especializada.

La ventaja principal es que el conocimiento utilizado por el sistema puede actualizarse modificando los documentos, sin necesidad de reentrenar el LLM.

## Construcción de un sistema RAG simple

A continuación construiremos un pipeline RAG básico compuesto por:

1. carga de documentos,
2. división en chunks,
3. generación de embeddings,
4. almacenamiento en una base vectorial,
5. búsqueda semántica,
6. y generación aumentada con contexto.

In [ ]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

/home/matthieu/Documents/trabajo/docencia/2026/MAD/sesion1/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
pdf_folder = "documentos"

documents = []

for filename in os.listdir(pdf_folder):

    if filename.endswith(".pdf"):

        filepath = os.path.join(pdf_folder, filename)

        print(f"Cargando: {filepath}")

        loader = PyPDFLoader(filepath)

        docs = loader.load()

        documents.extend(docs)

print()
print(f"Total de páginas cargadas: {len(documents)}")

Cargando: documentos/Productos-y-programas-comerciales-para-el-control-de-la-obesidad.pdf
Cargando: documentos/Cirugia-Bariatrica-Seleccion-y-estudio-preoperatorio.pdf

Total de páginas cargadas: 16


## Exploración inicial del contenido

Antes de construir el sistema RAG, es importante inspeccionar los documentos y comprender:
- calidad del texto,
- estructura,
- encabezados,
- tablas,
- y posibles problemas de extracción.

In [ ]:
print(documents[0].page_content[:10000])

Guía de práctica clínica para el manejo de la obesidad en adultos en Chile 1
MENSAJES CLAVE PARA EL 
PERSONAL DE SALUD
2022 adaptado por: Lamoza Pi, ii, iii; Andrade P .iv, v
El capítulo adaptado es de: Langlois MF, Freedhoff Y, Morin MP . 
Canadian Adult Obesity Clinical Practice Guidelines: 
Commercial Products and Programs in Obesity Management. 
(version 1, 2020). Disponible en: https://obesitycanada.ca/
guidelines/commercialproducts. © 2020 Obesity Canada.
i)  Clínica Colonial, Santiago, Chile.
ii)  Clínica Vespucio, Santiago, Chile.
iii)  Clínicas RCR Copiapó, Puerto Montt, Chile.
iv)  RedSalud Providencia, Santiago, Chile.
v)  Asociacion de Diabeticos de Chile (ADICH), Santiago, Chile.
Cómo citar este documento  
Productos y programas comerciales para el control de la obesidad. 
Adaptación de la guía de práctica clínica (Coalición chilena para el estudio 
de la obesidad, version 1, 2022) por Lamoza P , Andrade P . Capítulo 
adaptado de: Langlois MF, Freedhoff Y, Morin MP . Canad

## Chunking

Los documentos completos suelen ser demasiado largos para:
- búsqueda eficiente,
- embeddings,
- y contexto de modelos LLM.

Por esta razón, normalmente dividimos el texto en fragmentos más pequeños llamados *chunks*.

La elección del tamaño de chunk es una decisión importante de diseño:

- chunks demasiado pequeños pueden perder contexto,
- chunks demasiado grandes pueden introducir ruido,
- y chunks solapados (*overlap*) ayudan a preservar continuidad semántica.

En este ejemplo implementaremos un chunking simple basado en caracteres.

In [ ]:
!pip install langchain-text-splitters


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

chunks = splitter.split_documents(documents)

print(f"Total de chunks: {len(chunks)}")

Total de chunks: 161


In [ ]:
print(chunks[1].page_content)
print("---------------------")
print(chunks[2].page_content)

v)  Asociacion de Diabeticos de Chile (ADICH), Santiago, Chile.
Cómo citar este documento  
Productos y programas comerciales para el control de la obesidad. 
Adaptación de la guía de práctica clínica (Coalición chilena para el estudio 
de la obesidad, version 1, 2022) por Lamoza P , Andrade P . Capítulo 
adaptado de: Langlois MF, Freedhoff Y, Morin MP . Canadian Adult Obesity 
Clinical Practice Guidelines: Commercial Products and Programs in Obesity 
Management. (version 1, 2020). Disponible en: https://obesitycanada.ca/
guidelines/commercialproducts. © 2020 Obesity Canada.
Disponible en: guiasobesidadchile.com/prodcomerciales
Fecha de consulta [Fecha].
Productos y programas comerciales para 
el control de la obesidad  
• La industria comercial para la pérdida de peso es
---------------------
Fecha de consulta [Fecha].
Productos y programas comerciales para 
el control de la obesidad  
• La industria comercial para la pérdida de peso es 
enorme. Los médicos deben familiarizarse con la

El chunking es una de las decisiones más importantes en sistemas RAG.

En la práctica, pueden aparecer múltiples problemas:

- fragmentos truncados,
- encabezados repetidos,
- pérdida de contexto,
- chunks demasiado pequeños,
- chunks demasiado grandes,
- información duplicada,
- tablas mal extraídas,
- o fragmentos semánticamente incoherentes.

La calidad del retrieval depende fuertemente de la calidad de los chunks generados.

Intentemos mejorar un poco nuestro chunking separando el texto primero por párrafos,
luego por saltos de línea, luego por frases,luego por espacios,y SOLO al final por caracteres arbitrarios.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ],
    chunk_size=1000,
    chunk_overlap=150
)

In [ ]:
chunks = splitter.split_documents(documents)

print(f"Total de chunks: {len(chunks)}")

print(chunks[1].page_content)
print("---------------------")
print(chunks[2].page_content)

Total de chunks: 117
Adaptación de la guía de práctica clínica (Coalición chilena para el estudio 
de la obesidad, version 1, 2022) por Lamoza P , Andrade P . Capítulo 
adaptado de: Langlois MF, Freedhoff Y, Morin MP . Canadian Adult Obesity 
Clinical Practice Guidelines: Commercial Products and Programs in Obesity 
Management. (version 1, 2020). Disponible en: https://obesitycanada.ca/
guidelines/commercialproducts. © 2020 Obesity Canada.
Disponible en: guiasobesidadchile.com/prodcomerciales
Fecha de consulta [Fecha].
Productos y programas comerciales para 
el control de la obesidad  
• La industria comercial para la pérdida de peso es 
enorme. Los médicos deben familiarizarse con las ofertas 
comerciales para el manejo de la obesidad que existen en 
su entorno. Se han publicado criterios para evaluar si un 
programa comercial es seguro y potencialmente exitoso. Es 
decir, que ofrezca una combinación de nutrición, actividad 
física, y apoyo al cambio de comportamiento; con objetivos
-

En sistemas RAG reales, el chunking es un problema abierto de investigación e ingeniería.

Existen múltiples estrategias:
- tamaño fijo,
- chunking recursivo,
- segmentación por párrafos,
- segmentación semántica,
- segmentación basada en títulos,
- segmentación basada en embeddings,
- entre otras.

La estrategia elegida impacta directamente:
- calidad de retrieval,
- precisión,
- latencia,
- costo,
- y calidad final de respuestas.

## Embeddings y búsqueda semántica

Ahora transformaremos cada chunk en un embedding vectorial utilizando un modelo de NLP.

La idea principal es que:
- textos semánticamente similares tendrán vectores cercanos,
- permitiendo realizar búsqueda semántica y no solo búsqueda por palabras exactas.

Luego almacenaremos estos embeddings en una base vectorial (`ChromaDB`) y construiremos un *retriever* capaz de recuperar los chunks más relevantes para una pregunta dada.

El modelo `all-MiniLM-L6-v2` transforma texto en vectores numéricos (*embeddings*).

Por ejemplo:

```text
"actividad física"
```

→ vector numérico

```text
[0.12, -0.45, 0.91, ...]
```

La propiedad importante es que textos con significado similar producirán vectores cercanos entre sí.

El modelo `all-MiniLM-L6-v2`:
- es pequeño y rápido,
- funciona localmente,
- y es ampliamente utilizado para búsqueda semántica y sistemas RAG.

In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/home/matthieu/Documents/trabajo/docencia/2026/MAD/sesion1/.venv/lib/python3.12/site-packages/torch/cuda/__init__.py:187: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12020). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4025.06it/s]


`ChromaDB` permite:
- almacenar embeddings,
- indexarlos,
- y recuperar rápidamente los fragmentos más similares a una consulta.

Cada chunk es transformado automáticamente en embedding utilizando el modelo definido anteriormente.

El parámetro:

```python
persist_directory="chroma_db"
```

permite guardar la base vectorial en disco para reutilizarla posteriormente sin recalcular todos los embeddings.

In [ ]:
vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db"
)

El retriever es el componente encargado de:
- recibir una pregunta,
- realizar búsqueda semántica,
- y recuperar los chunks más relevantes.

El paráetro:

```python
k = 5
```

indica que queremos recuperar los 5 fragmentos más similares a la consulta realizada.

En un sistema RAG, estos chunks recuperados serán posteriormente utilizados como contexto para el LLM.

In [ ]:
retriever = vector_db.as_retriever(
    search_kwargs={"k": 5}
)

In [ ]:
query = """
A qué se debe mi apnea durante el sueño?
"""
#¿Es peligrosa la cirugía bariátrica?


results = retriever.invoke(query)

In [ ]:
for i, r in enumerate(results):

    print("=" * 80)
    print(f"Resultado {i+1}")
    print("=" * 80)

    print(r.page_content[:1500])

    print()

Resultado 1
negativamente al estado respiratorio de los candidatos a la cirugía 
bariátrica.50,51 
Apnea del sueño: No es inusual que las personas que viven con 
obesidad experimenten trastornos relacionados con el sueño, 
que pueden dar lugar a importantes afecciones respiratorias, 
cardiovasculares y neuropsiquiátricas. 52 La apnea obstructiva del 
sueño, un tipo de trastorno relacionado con el sueño, es el cese 
completo del flujo de aire (apnea) o la reducción significativa del

Resultado 2
negativamente al estado respiratorio de los candidatos a la cirugía 
bariátrica.50,51 
Apnea del sueño: No es inusual que las personas que viven con 
obesidad experimenten trastornos relacionados con el sueño, 
que pueden dar lugar a importantes afecciones respiratorias, 
cardiovasculares y neuropsiquiátricas. 52 La apnea obstructiva del 
sueño, un tipo de trastorno relacionado con el sueño, es el cese 
completo del flujo de aire (apnea) o la reducción significativa del

Resultado 3
Guía de prác

## Limitaciones reales de RAG

Los sistemas RAG no son mágicos.

Incluso utilizando embeddings modernos:
- algunos chunks irrelevantes pueden ser recuperados,
- algunos chunks relevantes pueden perderse,
- y el ranking semántico puede ser imperfecto.

La calidad final depende fuertemente de:
- calidad de documentos,
- chunking,
- embeddings,
- estrategia de retrieval,
- y reranking.

In [ ]:
context = "\n\n".join([doc.page_content for doc in results])

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

In [ ]:
prompt = f"""
Responde usando SOLO el contexto.

Contexto:
{context}

Pregunta:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

El contexto no proporciona información específica sobre si la cirugía bariátrica es peligrosa. Solo indica que la salud médica, mental, nutricional y funcional del paciente debe evaluarse antes de la cirugía, y que tras una evaluación, preparación y optimización adecuadas, con un perfil de riesgo perioperatorio aceptable, el paciente puede proceder a la cirugía bariátrica.


---

# Mejores prácticas para mejorar el Retrieval en un sistema RAG

Aunque un sistema RAG puede funcionar desde el inicio con embeddings + similarity search, la calidad del retrieval suele ser el factor más determinante en el rendimiento final.

A continuación se resumen las principales estrategias para mejorarlo, organizadas por impacto y nivel de madurez.


## 1.  Mejorar el chunking (impacto muy alto)

El chunking es uno de los factores más importantes en la calidad del retrieval.

### Recomendaciones base:
- Reducir tamaño de chunks:
  - `chunk_size`: 300–800 tokens
  - `chunk_overlap`: 10–20%
- Preferir chunking semántico:
  - por secciones
  - por subtítulos
  - por párrafos
- Evitar chunks que mezclen múltiples temas

### Multi-resolution chunking (opcional avanzado)

En sistemas más robustos, se pueden usar **distintos tamaños de chunks simultáneamente**:

####  Idea:
Representar el mismo documento a diferentes niveles de granularidad.

#### Ejemplo:
- chunks pequeños (200–400 tokens) → precisión local
- chunks medianos (500–800 tokens) → balance
- chunks grandes (1000–1500 tokens) → contexto global

#### Estrategias comunes:

**1. Multi-resolution indexing**
- Se indexan todos los niveles en el vector store

**2. Parent–child chunks (muy usado en producción)**
- Child chunks → embeddings (precisos)
- Parent chunk → contexto completo

Flujo:
1. retrieval sobre child chunks
2. expansión al parent chunk

Muy útil para documentos largos y preguntas complejas.

### Cuándo usarlo:
- ✔ cuando el retrieval base ya funciona bien
- ✔ cuando necesitas más robustez contextual
- ❌ no como primera optimización en un sistema roto

## 2. Limpieza de documentos (impacto muy alto y subestimado)

Los PDFs y documentos suelen contener ruido estructural que degrada embeddings.

### Elementos a eliminar:
- Headers y footers repetidos
- Números de página
- Disclaimers institucionales
- Texto duplicado o boilerplate
- Saltos de línea artificiales

### Beneficio:
Reduce ruido semántico y mejora coherencia de embeddings.


## 3. Mejorar modelos de embeddings (impacto alto)

Modelos modernos capturan mejor semántica y contexto.

### Modelos recomendados:
- `intfloat/multilingual-e5-base`
- `intfloat/multilingual-e5-large`
- `BAAI/bge-base-en-v1.5`
- `BAAI/bge-m3` (muy fuerte y versátil)


## 4. Hybrid Search (embeddings + BM25)

Combinar búsqueda semántica con búsqueda lexical.

### Ventajas:
- Embeddings → semántica
- BM25 → exact match (acrónimos, números, términos técnicos)


## 5. MMR (Max Marginal Relevance)

Técnica para mejorar diversidad de resultados.

### Problema:
Resultados muy similares entre sí.

### Solución:
Balance entre relevancia y diversidad.


## 6. Reranking (impacto muy alto en sistemas avanzados)

Pipeline típico:
1. Retrieval inicial (top 20–50)
2. Reranker
3. Selección final (top 5–10)

### Modelos:
- BGE reranker
- Cohere rerank
- Jina reranker


## 7. Uso de metadata y filtros

Permite restringir el espacio de búsqueda.

### Ejemplos:
- por documento
- por fecha
- por tipo
- por sección


## Conclusión

En sistemas RAG reales, el rendimiento no depende principalmente del LLM, sino del pipeline de retrieval.

Gran parte de la calidad proviene de:

- preprocessing de documentos
- chunking adecuado
- embeddings de calidad
- estrategias de búsqueda (MMR, hybrid search)
- reranking
- uso de metadata

Mejorar el retrieval suele ser más importante que cambiar el modelo generativo.

---

# Problemas en la fase de generación en RAG y cómo mejorarlos

Una vez que el retrieval funciona correctamente, la calidad del sistema RAG depende en gran medida de la fase de generación (LLM). En muchos casos, la calidad percibida de la respuesta no proviene del retrieval, sino de cómo el modelo utiliza el contexto recuperado.


## Observación clave

Incluso con un buen retrieval, pueden aparecer dos tipos de respuestas:

- Respuestas **muy fieles al contexto**, pero poco pedagógicas o difíciles de entender.
- Respuestas **muy claras y bien estructuradas**, pero menos ancladas en evidencia.

Esto muestra una tensión entre:

- precisión científica
- claridad y experiencia de usuario

# Problemas comunes en la generación

## 1. Falta de estructura en la respuesta

El modelo puede generar texto continuo sin organización clara.

**Problema:**
- respuesta difícil de leer
- sin separación entre conceptos

## 2. Bajo nivel pedagógico

Aunque la información sea correcta, puede ser:

- demasiado técnica
- poco explicada
- difícil para usuarios no expertos

## 3. Uso incompleto del contexto

El modelo puede:

- ignorar partes relevantes del contexto
- sintetizar demasiado agresivamente
- omitir evidencia importante

## 4. Desequilibrio entre generalización y fidelidad

- respuestas demasiado genéricas → parecen “ChatGPT base”
- respuestas demasiado literales → parecen extractos de papers


## 5. Falta de control del estilo

Sin instrucciones claras, el LLM decide libremente:

- tono
- nivel técnico
- estructura

# Cómo mejorar la fase de generación

## 1. Mejorar el prompt (factor más importante)

El prompt debe guiar explícitamente:

- estructura de la respuesta
- nivel de lenguaje
- uso del contexto

### Ejemplo recomendado:

```python
prompt = f"""
Eres un asistente experto en medicina.

Responde usando SOLO el contexto proporcionado.

INSTRUCCIONES:
- Primero explica de forma simple para un público general
- Luego presenta la evidencia del contexto
- Incluye limitaciones o incertidumbres si existen
- Finaliza con una conclusión clara
- Usa secciones bien estructuradas

Contexto:
{context}

Pregunta:
{query}
"""

In [ ]:
prompt = f"""
Responde usando SOLO el contexto.

Contexto:
{context}

Pregunta:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

El contexto no especifica directamente si la cirugía bariátrica es peligrosa, pero indica que antes de proceder a la cirugía se debe evaluar la salud médica, mental, nutricional y funcional del paciente, y que una vez realizada una adecuada evaluación, preparación y optimización, estableciendo un perfil de riesgo perioperatorio aceptable, el paciente puede proceder a la cirugía bariátrica. Esto implica que, si se cumplen estas condiciones, el riesgo puede ser controlado.


In [ ]:
prompt = f"""
Eres un asistente experto en medicina.

Responde usando SOLO el contexto proporcionado.

INSTRUCCIONES:
- Primero explica de forma simple para un público general
- Luego presenta la evidencia del contexto
- Incluye limitaciones o incertidumbres si existen
- Finaliza con una conclusión clara
- Usa secciones bien estructuradas

Contexto:
{context}

Pregunta:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

### Explicación sencilla para el público general

La cirugía bariátrica es un procedimiento que ayuda a perder peso en personas con obesidad severa y puede mejorar su calidad de vida. Como cualquier cirugía, tiene algunos riesgos, pero antes de realizarla, los médicos evalúan cuidadosamente la salud del paciente, incluyendo aspectos médicos, mentales, nutricionales y funcionales, para asegurarse de que el procedimiento sea seguro. Solo cuando se considera que el riesgo es aceptable, se procede a la cirugía.

### Evidencia del contexto

Según la información adaptada de las Guías de Canadá y la Coalición chilena para el estudio de la obesidad, la cirugía bariátrica puede ser eficaz y cambiar la vida de los pacientes con obesidad, siempre y cuando se realice una evaluación, preparación y optimización adecuada del paciente antes de la operación. Esto incluye un perfil de riesgo perioperatorio (el riesgo asociado a la cirugía y el periodo inmediato después de esta) que debe ser aceptable pa

La respuesta parece ser mejor...

---

# Upgrade del RAG: respuestas con citas (grounded generation)

Hasta ahora, nuestro sistema RAG recupera documentos relevantes y genera una respuesta.

Sin embargo, hay un problema importante:

> No sabemos exactamente de qué parte del contexto proviene cada afirmación.

En sistemas reales (medicina, legal, ciencia), esto es crítico.

Solución: **añadir trazabilidad mediante citas a fuentes**


##  Objetivo del upgrade

Queremos que el modelo:

- use el contexto recuperado
- indique explícitamente de qué documento proviene cada afirmación
- permita verificar la respuesta

In [ ]:
pdf_folder = "documentos"
documents = []

for filename in os.listdir(pdf_folder):

    if filename.endswith(".pdf"):

        filepath = os.path.join(pdf_folder, filename)
        print(f"Cargando: {filepath}")

        loader = PyPDFLoader(filepath)
        docs = loader.load()

        for doc in docs:
            doc.metadata["file_name"] = filename
            doc.metadata["source"] = filepath  # ruta completa
            documents.append(doc)

print(f"Total de páginas cargadas: {len(documents)}")

Cargando: documentos/Productos-y-programas-comerciales-para-el-control-de-la-obesidad.pdf
Cargando: documentos/Cirugia-Bariatrica-Seleccion-y-estudio-preoperatorio.pdf
Total de páginas cargadas: 16


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ],
    chunk_size=1000,
    chunk_overlap=150
)

In [ ]:
from langchain_core.documents import Document

chunks = []

for doc in documents:

    split_texts = splitter.split_text(doc.page_content)

    for i, chunk in enumerate(split_texts):

        chunks.append(
            Document(
                page_content=chunk,
                metadata={
                    "file_name": doc.metadata.get("file_name", "unknown"),
                    "source": doc.metadata.get("source", "unknown"),
                    "page": doc.metadata.get("page", None),
                    "chunk_id": i
                }
            )
        )

In [ ]:
print(chunks[0].metadata)
print(chunks[0].page_content[:300])

{'file_name': 'Productos-y-programas-comerciales-para-el-control-de-la-obesidad.pdf', 'source': 'documentos/Productos-y-programas-comerciales-para-el-control-de-la-obesidad.pdf', 'page': 0, 'chunk_id': 0}
Guía de práctica clínica para el manejo de la obesidad en adultos en Chile 1
MENSAJES CLAVE PARA EL 
PERSONAL DE SALUD
2022 adaptado por: Lamoza Pi, ii, iii; Andrade P .iv, v
El capítulo adaptado es de: Langlois MF, Freedhoff Y, Morin MP . 
Canadian Adult Obesity Clinical Practice Guidelines: 
Comme


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="chroma_db4"
)


retriever = vector_db.as_retriever(
    search_kwargs={"k": 5}
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14926.86it/s]


In [ ]:
query = "¿Es peligrosa la cirugía bariátrica?"

results = retriever.invoke(query)

In [ ]:
context = ""

for i, doc in enumerate(results):
    context += f"""
[Fuente {i+1}]
Documento: {doc.metadata.get("file_name")}
Página: {doc.metadata.get("page")}
Chunk: {doc.metadata.get("chunk_id")}

{doc.page_content}

---
"""

In [ ]:
prompt = f"""
Eres un asistente experto en medicina.

Responde usando SOLO el contexto.

INSTRUCCIONES:
- Explica de forma simple primero
- Incluye evidencia con fuentes
- Las fuentes incluyen el nombre del documento y la página, se ponen al final de la respuesta.
- Para citar la fuentes, pon un numero en el texto de la respuesta, y la información detallada de las fuentes al final
- Incluye limitaciones si existen
- Finaliza con una conclusión clara

Contexto:
{context}

Pregunta:
{query}
"""

response = llm.invoke(prompt)

print(response.content)

La cirugía bariátrica puede tener riesgos, como cualquier cirugía mayor, pero generalmente se considera segura cuando el paciente ha sido bien evaluado y preparado antes del procedimiento. Antes de realizar la cirugía, es fundamental evaluar la salud médica, mental, nutricional y funcional del paciente para asegurarse de que el perfil de riesgo perioperatorio sea aceptable. Esta preparación y optimización previa son claves para minimizar los riesgos asociados y garantizar el éxito de la intervención[3][4].

Esto significa que, aunque existen ciertos peligros potenciales, estos se reducen considerablemente con una adecuada selección y estudio preoperatorio. La cirugía bariátrica es una intervención eficaz y que puede cambiar la vida, especialmente para personas con obesidad grave que no han logrado bajar de peso con otros métodos.

Limitaciones: No se detalla en el contexto específico cuáles son los riesgos o complicaciones posibles, ni su frecuencia o gravedad. Tampoco se informa sobre